# Chest X-ray Classification Final Report
### Group 5 - Computing for Health and Medicine

This notebook contains the final data preprocessing pipeline, model architecture, training process, and evaluation visualizations for our Chest X-ray classification project.

## Executive Summary

### i. What worked best and why?

The CLAHE preprocessing made a big difference. This allows the model to detect faint pathological features without amplifying background noise. This is important for this dataset because the images are black and white with minimal contrast. Which makes it hard to detect abnormalities. We saw a small but consistent improvement across most classes. The loss function that we had the most success with was the asymmetric loss function. This is a variant of the binary cross-entropy loss function that is a specific version that is designed to handle imbalanced datasets. We have many more negatives than we do positives which helps prevent an imbalanced result in the model. Another thing that worked well was embedding the view position. This allowed the model to recognize which direction the image was taken from. We did this so the training would be able to recognize different feature adjustments depending on the view. 

### ii. What didn't help?

There were a handful of things that did not work to improve our results. Specifically the mixup augmentation, horizontal flips, and progressive resizing. We tried to create new training samples using mix up augmentation but failed because they were unrealistic for medical use. Horizontal flips failed because it is important to look at the images from the right direction. It will yield different results because the human body is not symmetrical so flipping them puts things in the wrong spot on the body. Which is all important information while training a model of this kind. Progressive resizing did not work because it ruined the details of the images making them harder to detect rather than easier.

### iii. What model will you likely submit for the final report?

We will use the Swin Transformer V2 with SimMIM pretraining. To improve on this architecture we will use a custom MLP head, class-specific attention pooling, and view position embeddings. The loss function we will use will be the asymmetric loss function. This model architecture with these improvements will give us the best possible results. 

## 1. Environment Setup & Constants

In [ ]:
import os
import glob
import math
import time
import datetime
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ExifTags
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer
from skmultilearn.model_selection import IterativeStratification
from torch.amp import autocast

# Import SwinV2 backbone from local file
from swin_transformer_v2 import SwinTransformerV2

ALL_CLASSES = [
    "Atelectasis","Cardiomegaly","Consolidation","Edema",
    "Effusion","Emphysema","Fibrosis","Hernia",
    "Infiltration","Mass","No Finding","Nodule",
    "Pleural_Thickening","Pneumonia","Pneumothorax",
]

NIH_CXR8_CUSTOM_MEAN = [0.5249, 0.5249, 0.5249]
NIH_CXR8_CUSTOM_STD  = [0.2622, 0.2622, 0.2622]

CLIP_LIMIT = 1.5
TILE_GRID_SIZE = 4
HORIZONTAL_FLIP_PROB = 0.5
ROTATION_DEGREES = 2.8
ROTATION_PROB = 0.5
JITTER_BRIGHTNESS = 0.08
JITTER_CONTRAST   = 0.08

VIEW_POSITION_SCALE = 0.35
FEATURE_DROPOUT    = 0.2
CLASSIFIER_DROPOUT = 0.1

print(f"Classes: {len(ALL_CLASSES)}")

## 2. Data Preprocessing Pipeline

Our pipeline includes orientation normalization and CLAHE (Contrast Limited Adaptive Histogram Equalization) to improve feature visibility.

In [ ]:
def normalize_cxr_image(img):
    """
    Normalize orientation of a CXR image using EXIF and pixel heuristics.
    """
    if img.mode != "L":
        img = img.convert("L")

    # Fix EXIF orientation
    try:
        exif = img._getexif()
        if exif is not None:
            orientation_key = next(k for k, v in ExifTags.TAGS.items() if v == "Orientation")
            orientation = exif.get(orientation_key)
            rotations = {3: 180, 6: 270, 8: 90}
            if orientation in rotations:
                img = img.rotate(rotations[orientation], expand=True)
    except Exception:
        pass

    arr = np.array(img)
    h, w = arr.shape

    # Lateral detection (skipped if too bright at edges)
    left_edge  = arr[:, :int(w * 0.15)].mean()
    right_edge = arr[:, int(w * 0.85):].mean()
    if max(left_edge, right_edge) > arr.mean() * 1.35:
        return None

    # Upside-down detection (diaphragm heuristic)
    if arr[:h//3].mean() > arr[-h//3:].mean() * 1.10:
        arr = np.flipud(arr)

    # Rib gradient check
    arr_f = arr.astype(np.float32)
    sobel_y = cv2.Sobel(arr_f, cv2.CV_32F, 0, 1, ksize=5)
    if np.abs(sobel_y[:h//2]).mean() < np.abs(sobel_y[h//2:]).mean() * 0.85:
        arr = np.flipud(arr)

    return arr.astype(np.uint8)

def apply_clahe(img_np):
    clahe = cv2.createCLAHE(clipLimit=CLIP_LIMIT, tileGridSize=(TILE_GRID_SIZE, TILE_GRID_SIZE))
    return clahe.apply(img_np)

## 3. Dataset & Data Loading

In [ ]:
class CXR8Dataset(Dataset):
    def __init__(self, df, labels, idx_array, transform, lookup):
        self.df        = df.iloc[idx_array].reset_index(drop=True)
        self.labels    = labels[idx_array]
        self.transform = transform
        self.lookup    = lookup

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        fname = self.df.loc[i, "Image Index"]
        path = self.lookup[fname]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        lbl = torch.tensor(self.labels[i], dtype=torch.float32)
        view_id = torch.tensor(self.df.loc[i, "view_id"], dtype=torch.long)
        return img, lbl, view_id

def make_train_tf(size):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(p=HORIZONTAL_FLIP_PROB),
        transforms.RandomApply([
            transforms.RandomRotation(ROTATION_DEGREES, interpolation=transforms.InterpolationMode.BILINEAR)
        ], p=ROTATION_PROB),
        transforms.ColorJitter(brightness=JITTER_BRIGHTNESS, contrast=JITTER_CONTRAST),
        transforms.ToTensor(),
        transforms.Normalize(NIH_CXR8_CUSTOM_MEAN, NIH_CXR8_CUSTOM_STD)
    ])

def make_value_tf(size):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(NIH_CXR8_CUSTOM_MEAN, NIH_CXR8_CUSTOM_STD)
    ])

## 4. Model Architecture

We use a Swin Transformer V2 backbone with class-specific attention pooling and view position embeddings.

In [ ]:
class ClassSpecificAttnPool(nn.Module):
    def __init__(self, C, num_classes):
        super().__init__()
        self.norm = nn.LayerNorm(C)
        self.query = nn.Linear(C, num_classes, bias=False)
        self.temp = nn.Parameter(torch.ones(1))

    def forward(self, feats):
        feats = self.norm(feats)
        attn = self.query(feats) / self.temp.clamp(min=0.1)
        attn = torch.softmax(attn, dim=1)
        pooled = torch.einsum("bnc,bnk->bkc", feats, attn)
        return pooled

class SwinWithView(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        C = backbone.norm.normalized_shape[0]
        backbone.head = nn.Identity()
        self.backbone = backbone
        self.num_classes = num_classes
        self.use_attention = True 

        self.attn_pool = ClassSpecificAttnPool(C, num_classes)
        self.view_embed = nn.Embedding(2, 32)
        self.view_mlp = nn.Sequential(
            nn.Linear(32, 128),
            nn.GELU(),
            nn.Linear(128, C * 2)
        )
        self.view_scale = nn.Parameter(torch.tensor(VIEW_POSITION_SCALE))

        self.head = nn.Sequential(
            nn.LayerNorm(C),
            nn.Dropout(FEATURE_DROPOUT),
            nn.Linear(C, 512),
            nn.GELU(),
            nn.Dropout(CLASSIFIER_DROPOUT),
            nn.Linear(512, 1),
        )

    def forward(self, x, view_id):
        feats = self.backbone.forward_features(x)
        v = self.view_mlp(self.view_embed(view_id))
        gamma, beta = v.chunk(2, dim=-1)
        scale = torch.sigmoid(self.view_scale) * 2.0
        feats = feats * (1 + scale * gamma.unsqueeze(1)) + beta.unsqueeze(1)

        if self.use_attention:
            pooled = self.attn_pool(feats)
        else:
            pooled = feats.mean(dim=1, keepdim=True).expand(-1, self.num_classes, -1)

        B, K, C = pooled.shape
        logits = self.head(pooled.reshape(B * K, C)).reshape(B, K)
        return logits

## 5. Training Process & Loss Function

We use Asymmetric Loss to handle the heavy class imbalance in the NIH CXR8 dataset.

In [ ]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_pos=0.2, gamma_neg=2.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.clip = clip
        self.eps = eps
        mask = torch.ones(len(ALL_CLASSES))
        mask[ALL_CLASSES.index("No Finding")] = 0.0
        self.register_buffer("loss_mask", mask)

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        xs_pos = probs
        xs_neg = (1 - probs + self.clip).clamp(max=1)

        loss = targets * torch.log(xs_pos.clamp(min=self.eps)) + (1 - targets) * torch.log(xs_neg.clamp(min=self.eps))
        loss = loss * self.loss_mask

        with torch.no_grad():
            pt = xs_pos * targets + (1 - xs_pos) * (1 - targets)
            gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
            focal_weight = (1 - pt) ** gamma

        loss *= focal_weight
        return -loss.sum() / (self.loss_mask.sum() * logits.shape[0])

def run_epoch(model, loader, criterion, optimizer=None, device='cuda'):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []

    with torch.set_grad_enabled(train):
        for imgs, lbls, views in tqdm(loader, desc="Training" if train else "Validation"):
            imgs, lbls, views = imgs.to(device), lbls.to(device), views.to(device)
            if train: optimizer.zero_grad()
            
            with autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(imgs, views)
                loss = criterion(logits, lbls)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            all_probs.append(logits.sigmoid().float().cpu().detach())
            all_labels.append(lbls.detach().cpu())

    return total_loss / len(loader.dataset), torch.cat(all_probs).numpy(), torch.cat(all_labels).numpy()

## 6. Visualization & Interpretability

GradCAM is used to visualize which regions of the X-ray the model is focusing on for a given prediction.

In [ ]:
class GradCAM:
    def __init__(self, model, device):
        self.model, self.device = model, device
        self._feats, self._grads = None, None
        target = model.backbone.layers[-1]
        self._fwd_hook = target.register_forward_hook(lambda m, i, o: setattr(self, '_feats', o[0] if isinstance(o, tuple) else o))
        self._bwd_hook = target.register_full_backward_hook(lambda m, gi, go: setattr(self, '_grads', go[0]))

    def __call__(self, img_tensor, class_idx, view_id=0):
        self.model.eval()
        x = img_tensor.unsqueeze(0).to(self.device)
        v = torch.tensor([view_id], dtype=torch.long, device=self.device)
        logits = self.model(x, v)
        self.model.zero_grad()
        logits[0, class_idx].backward()

        weights = self._grads.detach().mean(dim=1, keepdim=True)
        cam = torch.relu((weights * self._feats.detach()).sum(dim=-1)).squeeze(0)
        cam = cam.reshape(int(cam.shape[0]**0.5), -1).cpu().numpy()
        return (cam - cam.min()) / (cam.max() + 1e-8)

def visualize_prediction(model, img_tensor, img_np, class_idx, device, gradcam):
    cam = gradcam(img_tensor, class_idx)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_np, cmap='gray'); axes[0].set_title("Original")
    axes[1].imshow(img_np, cmap='gray')
    axes[1].imshow(heatmap, alpha=0.4); axes[1].set_title(f"GradCAM: {ALL_CLASSES[class_idx]}")
    plt.show()